# Testing the Force Components of CONTIGO

- [x] Gravatation Potential 
- [x] Third body acceleration
- [x] [Constants](https://ssd.jpl.nasa.gov/doc/Park.2021.AJ.DE440.pdf)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.integrate import cumulative_trapezoid
from scipy.integrate import cumulative_simpson

In [3]:
import contigo.config as config

In [4]:
print(config.DATA_DIR)
print(config.state)

D:\GitHub\contigo_edr\contigo\data
{'gmat_loaded': False, 'gmatpy': None, 'kernel_downloaded': False, 'pot_coef_loaded': False, 'pot_file': None, 'pot_clm': None, 'pot_slm': None, 'pot_r0': None, 'pot_GM': None}


In [6]:
from contigo.forces.grav_pot import GravPot
from contigo.forces.third_body_acc import ThirdBodyAcc
from contigo.forces.srp_acc import SRPGMATAcc

INFO:Using fallback library next to module: C:\Users\murph\miniconda3\envs\contigo\Lib\site-packages\spiceypy\utils\cspice.dll


In [7]:
sw_e = pd.read_hdf("./data/ESA_pod.hdf")
sw_o = pd.read_hdf("./data/ore_d.hdf")

efd_e = pd.read_hdf("./data/efd_esa_pod.hdf")
gmat_e = pd.read_hdf("./data/efd_gmat_ekf.hdf")

nmax = 100

spos = sw_e[['x','y','z']].to_numpy()
stime = sw_e['DateTime']
x = sw_e['x'].to_numpy()
y = sw_e['y'].to_numpy()
z = sw_e['z'].to_numpy()

lat = np.arctan2(z,np.sqrt(x*x+y*y))
lon = np.arctan2(y,x)

r = np.sqrt(x*x+y*y+z*z)

### Get the CONTIGO Gravatational Potential

In [8]:
gp_cont = GravPot(r=r,lat=lat,lon=lon,pot_file=r'EIGEN-2.gfc',lmax=100)
gp_cont.calc_pot()

INFO:Loading potential file D:\GitHub\contigo_edr\contigo\data\EIGEN-2.gfc


### Get the CONTIGO Third Body Acceleration from the Sun and Moon

In [10]:
tba_cont = ThirdBodyAcc(spos=spos,stime=stime,body=['SUN','MOON'],scale='GPS')  
tba_cont.calc_tba()

INFO:Kernel already loaded - D:\GitHub\contigo_edr\contigo\data\de440s.bsp
INFO:Kernel already loaded - D:\GitHub\contigo_edr\contigo\data\naif0012.tls
INFO:Kernel already loaded - D:\GitHub\contigo_edr\contigo\data\earth_latest_high_prec.bpc


### Get CONTIGO SRP Acceleration Via GMAT

In [ ]:
sc_state = sw_e[['x','y','z','vx','vy','vz']].to_numpy()
sc_time = sw_e['DateTime'].to_numpy()

scr = np.zeros_like(sc_state[:,0])
scm = np.zeros_like(sc_state[:,0])
sca = np.zeros_like(sc_state[:,0])
scr[:] = 1.8
scm[:] = 4.3e+02
sca[:] = 1

x = SRPGMATAcc(sc_state=sc_state, sc_time=sc_time, 
               sc_cr=scr, sc_srparea=sca, sc_mass=scm,
               apistartup="api_startup_file.txt",
               gmat_install="C:/Users/murph/GMAT_R2025a/")

x.calc_srp()

In [ ]:
srp_f = x.srp_acc_ecef
srp_i = x.srp_acc_eci

gmat_srp = sw_e[['srp_x', 'srp_y', 'srp_z']].to_numpy()

print(np.linalg.norm(srp_f,axis=1)[-40::10])
print(np.linalg.norm(srp_i,axis=1)[-40::10])
print(np.linalg.norm(gmat_srp,axis=1)[-40::10])


print(np.allclose(np.linalg.norm(srp_f,axis=1),np.linalg.norm(srp_i,axis=1)))
print(np.allclose(srp_i,gmat_srp))

### Compare CONTIGO Values to Orekit Values

Note that Orekit is in meters and CONTIGO is in Km's

In [ ]:
print(np.allclose(sw_o['earth_gp'].to_numpy(),gp_cont.gravpot*1000**2))
print(np.allclose(sw_o[['ecef_sn_ax','ecef_sn_ay','ecef_sn_az']], 
                  tba_cont.bd_acc[0,:,:]*1000))
print(np.allclose(sw_o[['ecef_mn_ax','ecef_mn_ay','ecef_mn_az']], 
                  tba_cont.bd_acc[1,:,:]*1000))


## Derive/Plot Effective Density

In [ ]:
# Sun and Moon Acceleratios Derived from Orekit
# Positions in ECEF
sn_af = sw_o[['ecef_sn_ax', 'ecef_sn_ay', 'ecef_sn_az']].to_numpy()
mn_af = sw_o[['ecef_mn_ax', 'ecef_mn_ay', 'ecef_mn_az']].to_numpy()

sn_af = tba_cont.bd_acc[0,:,:]*1000.
mn_af = tba_cont.bd_acc[1,:,:]*1000.

edr_pot = gp_cont.gravpot*1000**2

In [ ]:
print(sn_af[0,:])
print(mn_af[0,:])
print(srp_f[0,:]*1000)

In [ ]:
int_sec = (pd.DatetimeIndex(sw_e['DateTime']).to_julian_date()*86400.).to_numpy()
int_sec = int_sec-int_sec.min()

ecef_v = sw_e[['vx','vy','vz']].to_numpy()*1000
ecef_r = sw_e[['x','y','z']].to_numpy()*1000
ecef_xy2 = (sw_e['x'].to_numpy()*1000)**2+(sw_e['y'].to_numpy()*1000)**2
ecef_v2 = (ecef_v*ecef_v).sum(axis=1)
w2 = 5.3174941173225e-09

eci_v = sw_e[['eci_vx','eci_vy','eci_vz']].to_numpy()*1000
srp_a = sw_e[['srp_x','srp_y','srp_z']].to_numpy()*1000
srp_fms =  srp_f*1000.

a1 = (sn_af*ecef_v).sum(axis=1)
a2 = (mn_af*ecef_v).sum(axis=1)
a3 = (srp_fms*ecef_v).sum(axis=1) # this is a fudge factor that we can fix

# denomenator
B = (3.5)*(1.1)/(4.3e+02)
ecef_v3 = np.linalg.norm(ecef_v, axis=1)**3

In [ ]:
rolling = int(90*60/10)

edr = ecef_v2/2. - w2*ecef_xy2/2. - edr_pot - \
      cumulative_trapezoid(a1, int_sec, initial=0) - \
      cumulative_trapezoid(a2, int_sec,initial=0) - \
      cumulative_trapezoid(a3, int_sec,initial=0) # for exact comparison to orekit
edr = edr - ecef_v2[0]/2. - w2*ecef_xy2[0]/2. - edr_pot[0]

denom = B*cumulative_trapezoid(ecef_v3, int_sec,initial=0)

cont_edr = pd.DataFrame({'edr':edr, 'denom':denom})
cont_edr['DateTime'] = sw_e['DateTime']
cont_edr['edr_rolling'] = cont_edr['edr'].rolling(rolling).mean()  

In [ ]:
ax = cont_edr['edr'].plot(label='Contigo')
sw_o['edr'].plot(ax=ax, label='Orekit')
ax.legend()

ac = np.allclose(cont_edr['edr'].to_numpy(),sw_o['edr'].to_numpy())

print(f'Contigo and Orekit all close - {ac}')
print(cont_edr['edr'][0]/1000**2)
print(sw_o['edr'][0]/1000**2)


In [ ]:
intv = 5*90*60/10

ax = cont_edr.loc[0:intv,'edr'].plot(label='Contigo')
sw_o.loc[0:intv,'edr'].plot(ax=ax, label='Orekit')
ax.legend()

In [ ]:
ti = int(90*60/10.) 
cont_efd = pd.DataFrame()
cont_efd['efd'] = -2*(cont_edr['edr_rolling'].shift(ti)-cont_edr['edr_rolling'])/(cont_edr['denom'].shift(ti)-cont_edr['denom']) 
cont_efd['efd'] = cont_efd['efd']*(1000**3)
cont_efd['DateTime'] = cont_edr['DateTime']

In [ ]:
ax = efd_e.plot(x='DateTime',y=['tol_efd','efd_1'],label=['TOLEOS', 'ESA POD (ECI)'])
gmat_e.plot(x='DateTime',y='efd_0',label='GMAT EKF POD (ECI)', ax=ax)
cont_efd.plot(x='DateTime', y='efd', label='CONTIGO (ECEF)', ax=ax)

### Testing SRP a bit more

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from contigo.forces import srp_utils
srp_utils.setup_gmat("api_startup_file.txt","C:/Users/murph/GMAT_R2025a/")


In [ ]:
import contigo.config as config
from contigo.forces.srp_acc import SRPGMATAcc

In [ ]:
config.state['gmat_loaded']

In [ ]:
sw_e = pd.read_hdf("./data/ESA_pod.hdf")
sc_state = sw_e[['x','y','z','vx','vy','vz']].to_numpy()
sc_time = sw_e['DateTime'].to_numpy()

scr = np.zeros_like(sc_state[:,0])
scm = np.zeros_like(sc_state[:,0])
sca = np.zeros_like(sc_state[:,0])
scr[:] = 1.8
scm[:] = 4.3e+02
sca[:] = 1


In [ ]:
x = SRPGMATAcc(sc_state=sc_state, sc_time=sc_time, 
               sc_cr=scr, sc_srparea=sca, sc_mass=scm,
               apistartup="api_startup_file.txt",
               gmat_install="C:/Users/murph/GMAT_R2025a/")

In [ ]:
x.calc_srp()
srp_ecef, srp_eci = x.get_all_acc()

In [ ]:
srp_f = np.array(srp_ecef)
srp_i = np.array(srp_eci)

gmat_srp = sw_e[['srp_x', 'srp_y', 'srp_z']].to_numpy()

print(np.linalg.norm(srp_f,axis=1)[-40::10])
print(np.linalg.norm(srp_i,axis=1)[-40::10])
print(np.linalg.norm(gmat_srp,axis=1)[-40::10])


print(np.allclose(np.linalg.norm(srp_f,axis=1),np.linalg.norm(srp_i,axis=1)))
print(np.allclose(srp_i,gmat_srp))

### Test Changing Mass, Cr, and Area

In [ ]:
scr[::10] = scr[::10]*10
scr[::10] = scr[::10]*10
x = SRPGMATAcc(sc_state=sc_state, sc_time=sc_time, 
               sc_cr=scr, sc_srparea=sca, sc_mass=scm,
               apistartup="api_startup_file.txt",
               gmat_install="C:/Users/murph/GMAT_R2025a/")
x.calc_srp()
xa = x.get_ecef_acc()

scm[::10] = scm[::10]*10
y = SRPGMATAcc(sc_state=sc_state, sc_time=sc_time, 
               sc_cr=scr, sc_srparea=sca, sc_mass=scm,
               apistartup="api_startup_file.txt",
               gmat_install="C:/Users/murph/GMAT_R2025a/")
y.calc_srp()
ya = x.get_ecef_acc()


sca[::10] = sca[::10]*10
z = SRPGMATAcc(sc_state=sc_state, sc_time=sc_time, 
               sc_cr=scr, sc_srparea=sca, sc_mass=scm,
               apistartup="api_startup_file.txt",
               gmat_install="C:/Users/murph/GMAT_R2025a/")
z.calc_srp()
za = x.get_ecef_acc()

In [ ]:
print(srp_f[10,-3:])
print(xa[10,:])
print(ya[10,:])
print(za[10,:])
